# 宇宙論の「距離」はいくつもある

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/coxless/astro-sandbox/blob/main/cosmology/cosmic_distances.ipynb)

[distance/](../distance/) のノートブックでは、視差・距離指数・セファイド・超新星で「距離」を求めました。
近くの天体ではどれも同じ意味の距離ですが、宇宙が膨張しているので、遠くの天体では **何を測ったかによって距離の値が違ってきます**。
光が出たときと届いたときで宇宙の大きさが違うからです。

このノートブックでは

1. 代表的な距離の定義を並べ、赤方偏移とともにどれくらい食い違うかを見る
2. 遠くの天体ほど小さく見える……とは限らない(角径距離の最大値)
3. 「138 億歳の宇宙で、134 億光年先の銀河」とはどの距離のことか、観測できる宇宙の大きさ
4. 距離どうしの関係 $d_L = (1+z)^2 D_A$ を、超新星(Pantheon+)と BAO(DESI)の実データで確かめる

## 使うデータ

- 宇宙モデル: astropy に入っている Planck 2018 の ΛCDM(`Planck18`)
- Pantheon+(Scolnic et al. 2022, Brout et al. 2022): https://github.com/PantheonPlusSH0ES/DataRelease
- DESI DR2 BAO(DESI Collaboration 2025): https://github.com/CobayaSampler/bao_data

データのリポジトリにはライセンスの明記はありません。研究用に公開されたデータなので、使うときは論文を引用します。

## 0. 準備

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.cosmology import Planck18 as cosmo, z_at_value

SN_DATA_URL = (
    "https://raw.githubusercontent.com/PantheonPlusSH0ES/DataRelease/main/"
    "Pantheon%2B_Data/4_DISTANCES_AND_COVAR/Pantheon%2BSH0ES.dat"
)
BAO_URL = "https://raw.githubusercontent.com/CobayaSampler/bao_data/master/desi_bao_dr2/"
GLY_PER_MPC = (1 * u.Mpc).to(u.lyr).value / 1e9  # 1 Mpc = 0.00326 Gly
print(cosmo)

## 1. 距離の定義を並べる

赤方偏移 $z$ の天体について($E(z) = H(z)/H_0$、平坦な宇宙):

| 名前 | 式 | 意味 | どの観測で出てくるか |
| --- | --- | --- | --- |
| 共動距離 $D_C$ | $\dfrac{c}{H_0}\displaystyle\int_0^z \frac{dz'}{E(z')}$ | **今この瞬間** の、天体までの距離(定規を一瞬で並べたときの長さ) | BAO の $D_M$(平坦なら $D_M = D_C$) |
| 光度距離 $d_L$ | $(1+z)\,D_C$ | 明るさが $1/d^2$ で暗くなるとしたときの距離 | 超新星・セファイドなどの標準光源 |
| 角径距離 $D_A$ | $D_C / (1+z)$ | 見かけの大きさ $\theta = \ell / D$ から出した距離。光が出たときの天体までの距離でもある | BAO の角度、CMB のゆらぎの角度、銀河の見かけの大きさ |
| 光行距離 | $c\, t_{\mathrm{lookback}}(z)$ | 光が飛んでいた時間 × 光速。「○億光年先」とよく言われるもの | (ニュースなど) |

$d_L$ に $(1+z)$ が 2 回かかる(=$d_L = (1+z)^2 D_A$)のは、光子 1 個のエネルギーが $1/(1+z)$ に減ることと、光子が届く頻度が $1/(1+z)$ に減ることの 2 つによります。

In [ ]:
zz = np.logspace(-3, np.log10(1100), 500)
dc = cosmo.comoving_distance(zz).to(u.Mpc).value
dl = cosmo.luminosity_distance(zz).to(u.Mpc).value
da = cosmo.angular_diameter_distance(zz).to(u.Mpc).value
dt = (cosmo.lookback_time(zz) * 299_792.458 * u.km / u.s).to(u.Mpc).value  # 光行距離
naive = 299_792.458 * zz / cosmo.H0.value  # ハッブルの法則をそのまま使った d = cz / H0

fig, ax = plt.subplots(figsize=(8, 5))
for d, label, style in [
    (dl, "luminosity distance d_L", "-"),
    (dc, "comoving distance D_C (now)", "-"),
    (dt, "light-travel distance c t_lookback", "-"),
    (da, "angular diameter distance D_A", "-"),
    (naive, "cz / H0", ":"),
]:
    ax.plot(zz, d * GLY_PER_MPC, style, label=label)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylim(1e-2, 1e4)
ax.set_xlabel("redshift z")
ax.set_ylabel("distance [billion light years]")
ax.legend()
plt.show()

近く($z \ll 1$)ではどれも $cz/H_0$ と同じですが、$z = 1$ を超えると何倍も違ってきます。
どのくらいの $z$ から食い違いが目立つかを計算してみます。

In [ ]:
for frac in [0.01, 0.05, 0.10]:
    z_l = zz[np.argmax(np.abs(dl / dc - 1) > frac)]
    print(f"d_L と D_C が {frac:.0%} 違うのは z ≈ {z_l:.3f}(cz ≈ {299_792.458 * z_l:,.0f} km/s)")

$z = 0.01$(約 4 億光年)で、すでに 1% の違いになります。
超新星の距離の精度は 5% 程度なので、ハッブル定数を測る $0.023 < z < 0.15$ の範囲では、どの距離かをきちんと区別する必要があります
([distance/sn_ia_hubble.ipynb](../distance/sn_ia_hubble.ipynb) で $d_L$ の近似式に $z$ の 2 次まで入れていたのはこのためです)。

## 2. 遠くの天体ほど小さく見える……とは限らない

大きさ $\ell$ の天体の見かけの角度は $\theta = \ell / D_A$ です。
上の図の $D_A$(赤)は $z \approx 1.6$ で最大になり、それより遠いと **小さくなっていきます**。
遠くの天体の光は、宇宙がまだ小さかった頃に、今よりずっと近い場所から出たからです。

直径 30 kpc の銀河(天の川銀河くらい)と、BAO の物差し $r_d \approx 147$ Mpc(共動の長さなので、光が出たときの実際の大きさは $r_d / (1+z)$)の見かけの大きさを計算します。

In [ ]:
i_max = np.argmax(da)
print(f"角径距離が最大になるのは z = {zz[i_max]:.2f}, D_A = {da[i_max] * GLY_PER_MPC * 10:.0f} 億光年")

GALAXY_KPC = 30.0
RD_MPC = 147.09
theta_gal = (GALAXY_KPC / 1000 / da) * 206_265  # [秒角]
theta_bao = np.degrees(RD_MPC / (1 + zz) / da)  # [度]。= rd / D_M

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(zz, theta_gal)
axes[0].set_ylabel(f"angular size of a {GALAXY_KPC:.0f} kpc galaxy [arcsec]")
axes[1].plot(zz, theta_bao)
axes[1].set_ylabel("angular size of the BAO scale r_d [deg]")
for ax in axes:
    ax.axvline(zz[i_max], c="gray", ls=":")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("redshift z")
axes[0].set_xlim(0.01, 1100)
axes[1].set_xlim(0.1, 1100)
plt.tight_layout()
plt.show()

z_star = 1089.9  # 光子が離れた赤方偏移(Planck 2018)
rs_star = 144.4  # そのときの音響地平線(共動)[Mpc], Planck 2018
theta_star = rs_star / cosmo.comoving_distance(z_star).to(u.Mpc).value
print(f"z = 0.1 の銀河: {GALAXY_KPC / 1000 / cosmo.angular_diameter_distance(0.1).value * 206265:.1f} 秒角, "
      f"z = 1.6: {GALAXY_KPC / 1000 / da[i_max] * 206265:.1f} 秒角, z = 10: {GALAXY_KPC / 1000 / cosmo.angular_diameter_distance(10).value * 206265:.1f} 秒角")
print(f"CMB の音響ピークの角度 θ* = r_s / D_M(z*) = {np.degrees(theta_star):.3f}°(Planck の測定: 0.5965°)")

- $z \gtrsim 1.6$ では、遠くの銀河ほど(同じ大きさなら)見かけは大きくなります。それでも見つけにくいのは、表面輝度が $(1+z)^{-4}$ で急激に暗くなるからです(トールマンの減光。$d_L^2 / D_A^2 = (1+z)^4$ から)
- CMB のゆらぎに刻まれた音波の大きさ(約 0.6°)は、この $D_M$ の計算でぴったり再現されます。この角度は Planck が 0.03% の精度で測っていて、宇宙論でいちばんよく測られている数の 1 つです

## 3. 「134 億光年先」とはどの距離か

ニュースで「○億光年先の銀河」と言うときは、たいてい **光行距離**(光が飛んでいた時間 × 光速)です。
その銀河が **今** どこにいるか(共動距離)は、そのあと宇宙が膨張したぶんずっと遠くになっています。

いくつかの天体について比べます。

In [ ]:
OBJECTS = [
    ("Virgo cluster", 0.0036),
    ("SN Ia at the SH0ES limit", 0.023),
    ("SN Ia at z = 0.5", 0.5),
    ("most distant SN in Pantheon+", 2.26),
    ("GN-z11 (Bunker et al. 2023)", 10.60),
    ("JADES-GS-z14-0 (Carniani et al. 2024)", 14.32),
    ("CMB (last scattering)", 1089.9),
]
rows = []
for name, z in OBJECTS:
    rows.append({
        "object": name,
        "z": z,
        "lookback [Gyr]": cosmo.lookback_time(z).to(u.Gyr).value,
        "age then [Myr]": cosmo.age(z).to(u.Myr).value,
        "light-travel [Gly]": cosmo.lookback_time(z).to(u.Gyr).value,  # 光行距離 [億光年 ÷ 10] = ルックバック時間 [Gyr]
        "comoving now [Gly]": cosmo.comoving_distance(z).to(u.lyr).value / 1e9,
        "distance at emission [Gly]": cosmo.angular_diameter_distance(z).to(u.lyr).value / 1e9,
        "luminosity [Gly]": cosmo.luminosity_distance(z).to(u.lyr).value / 1e9,
    })
table = pd.DataFrame(rows)
table.round(3)

In [ ]:
# 粒子地平線 = z → ∞ までの共動距離。z が大きいところまで積分するので、ln(1+z) について等間隔に台形公式で積分する
ln1pz = np.linspace(0, np.log(1e12), 200_001)
z_int = np.expm1(ln1pz)
horizon = (cosmo.hubble_distance * np.trapezoid((1 + z_int) * cosmo.inv_efunc(z_int), ln1pz)).to(u.lyr).value / 1e9
print(f"宇宙の年齢: {cosmo.age(0).to(u.Gyr).value:.2f} Gyr")
print(f"観測できる宇宙の半径(粒子地平線、今の距離): {horizon:.1f} Gly(約 {horizon * 10:.0f} 億光年)")
jades = table.set_index("object").loc["JADES-GS-z14-0 (Carniani et al. 2024)"]
print(f"JADES-GS-z14-0: 光は {jades['lookback [Gyr]'] * 10:.0f} 億年前に出た(宇宙誕生から {jades['age then [Myr]']:.0f} 百万年後)。"
      f"そのときの距離は {jades['distance at emission [Gly]'] * 10:.0f} 億光年、今の距離は {jades['comoving now [Gly]'] * 10:.0f} 億光年")

- 宇宙の年齢は 138 億年なので、光行距離は 138 億光年を超えません。でも、観測できる宇宙の半径(今の距離)は約 **460 億光年** あります
- 最も遠い銀河の 1 つ JADES-GS-z14-0 の光は、宇宙が生まれて 3 億年ほどのときに、私たちから **わずか 20 数億光年** の場所で出て、
  それから 135 億年かかって届きました。その間に宇宙が膨張し、銀河は今 **約 340 億光年** 先にいます
- 光を出したときの距離(= 角径距離)は、$z$ が大きいとむしろ小さくなります。CMB の光は、今の私たちから 4000 万光年ほどのところで出た光です

## 4. 距離の関係 $d_L = (1+z)^2 D_A$ を実データで確かめる

$d_L$ と $D_A$ の関係(エザリントンの相反関係、距離双対性)は、
「光子の数が途中で減らない」「光が測地線に沿って進む」ことだけから導かれ、どんな宇宙モデルでも成り立ちます。
これが破れていたら、光が途中で吸収される(宇宙の塵など)か、光子がほかの粒子に変わる、といった新しい物理を意味します。

観測では

- **超新星** は光度距離 $d_L$ を測る(ただし $M_B$ と $H_0$ が分からないと、全体の目盛りが決まらない)
- **BAO** は $D_M / r_d = (1+z) D_A / r_d$ を測る($r_d$ が分からないと、全体の目盛りが決まらない)

ので、比 $\eta(z) = \dfrac{d_L}{(1+z)\,D_M}$ は **定数倍を除いて** 測れます。関係が正しければ、$\eta$ は $z$ によらず一定のはずです。

BAO の各赤方偏移の近くにある超新星の明るさを平均して、その赤方偏移での $d_L$(の相対値)を出します。

In [ ]:
raw = pd.read_csv(SN_DATA_URL, sep=r"\s+")
sn = raw[raw["IS_CALIBRATOR"] == 0].groupby("CID").agg(
    zHD=("zHD", "mean"), zHEL=("zHEL", "mean"), m_b=("m_b_corr", "mean"), m_b_err=("m_b_corr_err_DIAG", "mean")
).reset_index()

bao = pd.read_csv(BAO_URL + "desi_gaussian_bao_ALL_GCcomb_mean.txt", sep=r"\s+", comment="#", header=None,
                  names=["z", "value", "quantity"])
bao["err"] = np.sqrt(np.diag(np.loadtxt(BAO_URL + "desi_gaussian_bao_ALL_GCcomb_cov.txt")))
dm = bao[bao["quantity"] == "DM_over_rs"].reset_index(drop=True)
dm

赤方偏移の幅 $\Delta z$ の中の超新星について、それぞれの等級を BAO の赤方偏移 $z_{\mathrm{BAO}}$ での値に直してから平均します。
幅の中での $d_L$ の変化は小さいので、その分の補正には(どのモデルでもほぼ同じになる)Planck の ΛCDM を使います。

In [ ]:
HALF_WIDTH = 0.10
rows = []
for _, b in dm.iterrows():
    sel = sn[np.abs(sn["zHD"] - b["z"]) < HALF_WIDTH]
    if len(sel) < 5:
        continue
    # 各超新星の等級を z_BAO での値に直す(幅の中での d_L の変化を Planck ΛCDM で補正)
    shift = 5 * np.log10(cosmo.luminosity_distance(b["z"]).value / cosmo.luminosity_distance(sel["zHD"].to_numpy()).value)
    m_at_z = sel["m_b"].to_numpy() + shift
    w = 1 / sel["m_b_err"].to_numpy() ** 2
    m_mean = np.sum(w * m_at_z) / np.sum(w)
    m_err = np.sqrt(1 / np.sum(w))
    # d_L ∝ 10^(m/5)、D_M ∝ (D_M/r_d)
    log_eta = m_mean / 5 - np.log10((1 + b["z"]) * b["value"])
    log_eta_err = np.hypot(m_err / 5, b["err"] / b["value"] / np.log(10))
    rows.append(dict(z=b["z"], n_sn=len(sel), log_eta=log_eta, log_eta_err=log_eta_err))
eta = pd.DataFrame(rows)

# 定数倍は分からないので、重み付き平均が 1 になるように規格化する
w = 1 / eta["log_eta_err"] ** 2
eta["eta"] = 10 ** (eta["log_eta"] - np.sum(w * eta["log_eta"]) / np.sum(w))
eta["eta_err"] = eta["eta"] * np.log(10) * eta["log_eta_err"]
eta

In [ ]:
chi2 = np.sum(((eta["eta"] - 1) / eta["eta_err"]) ** 2)
print(f"η = 一定 としたときの χ² = {chi2:.1f}(自由度 {len(eta) - 1})")

# η(z) = (1+z)^ε と書いて、ε を当てはめる(ε = 0 なら関係が成り立つ)
x = np.log10(1 + eta["z"])
coef, cov = np.polyfit(x, np.log10(eta["eta"]), 1, w=1 / (eta["eta_err"] / eta["eta"] / np.log(10)), cov="unscaled")
eps, eps_err = coef[0], np.sqrt(cov[0, 0])
print(f"η ∝ (1+z)^ε: ε = {eps:.3f} ± {eps_err:.3f}")

plt.figure(figsize=(7, 4))
plt.errorbar(eta["z"], eta["eta"], yerr=eta["eta_err"], fmt="o")
zz_e = np.linspace(0.4, 1.6, 50)
plt.plot(zz_e, 10 ** np.polyval(coef, np.log10(1 + zz_e)), c="tab:red", lw=1, label=f"fit (1+z)^ε, ε = {eps:.2f} ± {eps_err:.2f}")
plt.axhline(1, c="k", ls="--", lw=1, label="distance duality (η = const)")
for k in [1, -1]:  # 比較用: ε = ±0.1(z = 0.8 で 1 になるように)
    plt.plot(zz_e, ((1 + zz_e) / 1.8) ** (0.1 * k), c="gray", ls=":", lw=1, label="ε = ±0.1" if k == 1 else None)
plt.xlabel("redshift z")
plt.ylabel("η = d_L / ((1+z) D_M)  (normalized)")
plt.legend(loc="upper left")
plt.show()

超新星と BAO は、まったく違う方法(明るさと、見かけの大きさ)で距離を測っていますが、$\eta$ は誤差の範囲で一定です
(χ² が自由度と同じくらい。$z = 0.5$ と $0.7$ の 2 点は 1.5% ほどの精度で一致)。
$\varepsilon$ は 0 から 2σ 以内で、$z \approx 0.9$ より遠くは超新星の数が少ないので、まだ誤差が大きいです。

> ここでは超新星を幅 ±0.1 で平均するという簡単な方法を使いました。共分散行列や BAO の全データを使った本格的な解析では、$\varepsilon$ の誤差は数 % まで小さくなり、0 と矛盾しない結果が出ています。

## まとめ

| 距離 | 一言でいうと | 測る方法 |
| --- | --- | --- |
| 共動距離 $D_C$ | 今の距離 | BAO の $D_M$ |
| 光度距離 $d_L = (1+z) D_C$ | 明るさから出す距離 | 超新星、セファイド |
| 角径距離 $D_A = D_C / (1+z)$ | 見かけの大きさから出す距離 = 光を出したときの距離 | BAO の角度、CMB |
| 光行距離 $c\,t_{\mathrm{lookback}}$ | 光が飛んでいた時間 | (ニュースの「○億光年」) |

- $z \approx 0.01$ ですでに 1% 違う。$z > 1$ では何倍も違う
- 角径距離は $z \approx 1.6$ で最大になり、それより遠い天体は大きく見えてくる
- 観測できる宇宙の半径は、今の距離で約 460 億光年
- $d_L = (1+z)^2 D_A$ は、超新星と BAO の実データで誤差の範囲で成り立つ($z < 0.7$ では約 1.5% の精度)

## 発展メモ

- 平坦でない宇宙($\Omega_k \neq 0$)にすると、$D_M$ と $D_C$ がずれる。astropy の `LambdaCDM` で $\Omega_k = \pm 0.1$ にして比べる
- 4 章の幅 `HALF_WIDTH` を変えると $\eta$ はどう変わるか
- 天体が「今」光速より速く遠ざかっている距離(ハッブル球)と、今出た光が二度と届かない距離(事象の地平線)を計算する

## 参考・クレジット

- Hogg (1999), *Distance measures in cosmology*, arXiv:astro-ph/9905116
- Etherington (1933), *On the Definition of Distance in General Relativity*, Phil. Mag. 15, 761
- Planck Collaboration (2020), *Planck 2018 results. VI. Cosmological parameters*, A&A 641, A6
- Scolnic et al. (2022), ApJ 938, 113 / Brout et al. (2022), ApJ 938, 110(Pantheon+)
- DESI Collaboration (2025), *DESI DR2 Results II*, arXiv:2503.14738
- Bunker et al. (2023), *JADES NIRSpec Spectroscopy of GN-z11*, A&A 677, A88
- Carniani et al. (2024), *Spectroscopic confirmation of two luminous galaxies at a redshift of 14*, Nature 633, 318
- Davis & Lineweaver (2004), *Expanding Confusion*, PASA 21, 97